In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("cleaned_results.csv")
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')

df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result,goal_diff
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,Draw,0.0
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,Home Win,2.0
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,Home Win,1.0
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,Draw,0.0
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,Home Win,3.0


In [ ]:
home_df = df.rename(columns={
    'home_team': 'team',
    'away_team': 'opponent',
    'home_score': 'team_goals',
    'away_score': 'opponent_goals'
})

away_df = df.rename(columns={
    'away_team': 'team',
    'home_team': 'opponent',
    'away_score': 'team_goals',
    'home_score': 'opponent_goals'
})

matches = pd.concat([home_df, away_df], ignore_index=True)

In [ ]:
def get_team_result(row):
    if row['team_goals'] > row['opponent_goals']:
        return "Win"
    elif row['team_goals'] < row['opponent_goals']:
        return "Loss"
    else:
        return "Draw"

matches['team_result'] = matches.apply(get_team_result, axis=1)

In [ ]:
matches['win'] = matches['team_result'].apply(lambda x: 1 if x == "Win" else 0)

In [ ]:
matches = matches.sort_values('date')

matches['team_avg_goals_last5'] = (
    matches.groupby('team')['team_goals']
    .transform(lambda x: x.shift().rolling(5).mean())
)

In [ ]:
matches['team_avg_conceded_last5'] = (
    matches.groupby('team')['opponent_goals']
    .transform(lambda x: x.shift().rolling(5).mean())
)

In [ ]:
matches['team_win_rate_last5'] = (
    matches.groupby('team')['win']
    .transform(lambda x: x.shift().rolling(5).mean())
)

In [ ]:
opponent_stats = matches[['date', 'team', 'team_avg_goals_last5',
                          'team_avg_conceded_last5', 'team_win_rate_last5']]

opponent_stats = opponent_stats.rename(columns={
    'team': 'opponent',
    'team_avg_goals_last5': 'opp_avg_goals_last5',
    'team_avg_conceded_last5': 'opp_avg_conceded_last5',
    'team_win_rate_last5': 'opp_win_rate_last5'
})

matches = matches.merge(
    opponent_stats,
    on=['date', 'opponent'],
    how='left'
)

In [ ]:
matches['goals_diff_last5'] = matches['team_avg_goals_last5'] - matches['opp_avg_goals_last5']
matches['conceded_diff_last5'] = matches['team_avg_conceded_last5'] - matches['opp_avg_conceded_last5']
matches['win_rate_diff'] = matches['team_win_rate_last5'] - matches['opp_win_rate_last5']

In [ ]:
matches_model = matches.dropna()

In [ ]:
matches_model.to_csv("features_dataset.csv", index=False)

In [12]:
from google.colab import files
files.download("features_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>